# ☕ ICO Coffee Data — Full Cleaning, Sentinel-Bug Deep Dive & Merging Pipeline

This notebook combines two things that were previously separate:
1. The full 7-file cleaning + merging pipeline (`clean_ico_coffee_data.py`)
2. A detailed walkthrough of one specific issue found along the way — the sentinel-value bug in `Coffee_export.csv` — including the proof that the real values are unrecoverable, and an optional ratio-based estimate for them

**The 4 known issues this notebook fixes:**
1. **Corrupted sentinel values** (`-2147483648`, a classic 32-bit integer overflow / null placeholder) in `Coffee_export.csv` → converted to `NaN` (walked through in detail in Step 4 below)
2. **Inconsistent country-name whitespace** (e.g. `"   Austria"`) → stripped
3. **Mismatched country sets across files** (55 producing countries vs. 35 importing countries) → handled with an outer union instead of forcing an inner join, which would silently drop real data
4. **Crop-year vs. calendar-year labels** (`"1990/91"` vs. `"1990"`) → both mapped to a single clean integer `Year` column (crop year → its starting year)

**Output:** one tidy long-format table (`Country | Coffee type | Metric | Year | value | is_imputed`), plus one wide-format CSV per metric.

## Step 0 — Setup

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 2.3.3
numpy: 2.3.5


## Step 1 — Upload the 7 raw ICO CSV files

Select all 7 at once:

- `Coffee_production.csv`
- `Coffee_domestic_consumption.csv`
- `Coffee_export.csv`
- `Coffee_import.csv`
- `Coffee_re_export.csv`
- `Coffee_importers_consumption.csv`
- `Coffee_green_coffee_inventorie.csv`

Colab's `files.upload()` saves them into the notebook's working directory automatically.

In [2]:
try:
    from google.colab import files
    print("Running in Colab — opening the file picker (select all 7 CSVs at once)...")
    uploaded = files.upload()
    print(f"\nUploaded {len(uploaded)} file(s):", list(uploaded.keys()))
except ImportError:
    print("Not running in Colab — skipping the upload widget.")
    print("Make sure the 7 CSVs are already present in INPUT_DIR (set in the next cell).")

Not running in Colab — skipping the upload widget.
Make sure the 7 CSVs are already present in INPUT_DIR (set in the next cell).


## Step 2 — Config

In [3]:
INPUT_DIR = Path(".")       # files.upload() above saves here in Colab
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SENTINEL = -2147483648  # -2^31, a classic int32 overflow / null placeholder

# File name -> (tidy metric name, whether it has a "Coffee type" column)
FILES = {
    "Coffee_production.csv": ("production_kg", True),
    "Coffee_domestic_consumption.csv": ("domestic_consumption_kg", True),
    "Coffee_export.csv": ("export_kg", False),
    "Coffee_import.csv": ("import_kg", False),
    "Coffee_re_export.csv": ("re_export_kg", False),
    "Coffee_importers_consumption.csv": ("importer_consumption_kg", False),
    "Coffee_green_coffee_inventorie.csv": ("green_coffee_inventory_kg", False),
}

print("Looking for files in:", INPUT_DIR.resolve())
for fname in FILES:
    exists = (INPUT_DIR / fname).exists()
    print(f"  {'✅' if exists else '❌'} {fname}")

Looking for files in: C:\Users\KrissW Laptop\Downloads\initial data cleaning
  ✅ Coffee_production.csv
  ✅ Coffee_domestic_consumption.csv
  ✅ Coffee_export.csv
  ✅ Coffee_import.csv
  ✅ Coffee_re_export.csv
  ✅ Coffee_importers_consumption.csv
  ✅ Coffee_green_coffee_inventorie.csv


## Step 3 — Helper: convert year-like column names to a clean int year

Handles both column styles found in the raw files:
- `"1990"` → `1990`
- `"1990/91"` (crop year) → `1990` (its starting calendar year)

In [4]:
def year_col_to_int(col: str) -> int | None:
    """
    '1990'    -> 1990
    '1990/91' -> 1990   (crop year -> its starting calendar year)
    Anything else (e.g. 'Country', 'Total_export') -> None
    """
    col = col.strip()
    if re.fullmatch(r"\d{4}", col):
        return int(col)
    m = re.fullmatch(r"(\d{4})/\d{2}", col)
    if m:
        return int(m.group(1))
    return None

assert year_col_to_int("1990") == 1990
assert year_col_to_int("1990/91") == 1990
assert year_col_to_int("Country") is None
print("year_col_to_int: OK")

year_col_to_int: OK


## Step 4 — Deep dive: the sentinel-value bug in `Coffee_export.csv`

Before writing the general-purpose cleaning function, it's worth understanding *exactly* what this bug is and whether the true values can be recovered — this directly shapes how `load_and_tidy` handles it below.

### 4a. Find it

In [5]:
export_path = INPUT_DIR / "Coffee_export.csv"
raw_export = pd.read_csv(export_path)
raw_export["Country"] = raw_export["Country"].astype(str).str.strip()

export_year_cols = [c for c in raw_export.columns if c not in ("Country", "Total_export")]
sentinel_mask = raw_export[export_year_cols].eq(SENTINEL)

affected = sentinel_mask.stack()
affected = affected[affected].index.tolist()
print(f"{len(affected)} cell(s) contain the sentinel value -2147483648:")
for row_idx, year_col in affected:
    print(f"  {raw_export.loc[row_idx, 'Country']} — {year_col}")

3 cell(s) contain the sentinel value -2147483648:
  Brazil — 2014
  Brazil — 2015
  Brazil — 2019


`-2147483648` is exactly **−2³¹** — the most negative value a signed 32-bit integer can hold. Seeing this exact number in trade data is a classic tell for an integer-overflow or NULL-placeholder bug upstream.

### 4b. Can the real values be recovered from this file?

The file has its own `Total_export` column. If that total was computed *before* the corruption happened, the real values might still be reconstructable from it. Let's check the arithmetic.

In [6]:
brazil_row = raw_export[raw_export.Country == "Brazil"].iloc[0]
good_years = [c for c in export_year_cols if brazil_row[c] != SENTINEL]

good_sum = brazil_row[good_years].sum()
reported_total = brazil_row["Total_export"]

print(f"Sum of the 27 good years:  {good_sum:,.0f}")
print(f"Reported Total_export:     {reported_total:,.0f}")
print(f"Difference:                {reported_total - good_sum:,.0f}")
print(f"3 x SENTINEL:              {3 * SENTINEL:,.0f}")
print()
print("Difference == 3 x SENTINEL:", (reported_total - good_sum) == 3 * SENTINEL)

Sum of the 27 good years:  40,250,160,000
Reported Total_export:     33,807,709,056
Difference:                -6,442,450,944
3 x SENTINEL:              -6,442,450,944

Difference == 3 x SENTINEL: True


**The difference matches `3 × SENTINEL` exactly**, which means:

```
Reported Total_export = sum(27 good years) + 3 × (−2,147,483,648)
```

The total was computed **after** the corruption already happened, so the real 2014/2015/2019 values are genuinely gone from this file — not just hidden. That rules out "reverse-engineer it from the total," which settles how `load_and_tidy` should treat it below: convert to `NaN`, never trust it as a real number.

## Step 5 — `load_and_tidy`: load one file and apply fixes #1, #2, #4

Now that fix #1 is well understood, here's the general-purpose function that applies it (along with #2 and #4) to any of the 7 files.

In [7]:
def load_and_tidy(path: Path, metric_name: str, has_coffee_type: bool) -> pd.DataFrame:
    """Load one raw ICO CSV and return a tidy long-format DataFrame."""
    df = pd.read_csv(path)

    # --- Fix #2: strip whitespace from country names ---
    df["Country"] = df["Country"].astype(str).str.strip()
    if has_coffee_type and "Coffee type" in df.columns:
        df["Coffee type"] = df["Coffee type"].astype(str).str.strip()
    else:
        df["Coffee type"] = np.nan

    year_map = {c: year_col_to_int(c) for c in df.columns}
    year_cols = [c for c, y in year_map.items() if y is not None]

    # --- Fix #1: replace corrupted sentinel values with NaN (see Step 4) ---
    df[year_cols] = df[year_cols].apply(pd.to_numeric, errors="coerce")
    df[year_cols] = df[year_cols].replace(SENTINEL, np.nan)

    # --- Fix #4: melt to long format with a single clean Year column ---
    long_df = df.melt(
        id_vars=["Country", "Coffee type"],
        value_vars=year_cols,
        var_name="raw_year_label",
        value_name="value",
    )
    long_df["Year"] = long_df["raw_year_label"].map(year_map)
    long_df["Metric"] = metric_name
    long_df = long_df.drop(columns="raw_year_label")
    long_df = long_df.dropna(subset=["value"])

    return long_df[["Country", "Coffee type", "Metric", "Year", "value"]]

# Sanity-check on Coffee_export.csv: the 3 sentinel rows should now be gone
_test = load_and_tidy(export_path, "export_kg", False)
_brazil_years = set(_test[_test.Country == "Brazil"]["Year"])
print("Brazil years present after cleaning (2014/2015/2019 should be missing):")
print(sorted(_brazil_years))

Brazil years present after cleaning (2014/2015/2019 should be missing):
[1990.0, 1991.0, 1992.0, 1993.0, 1994.0, 1995.0, 1996.0, 1997.0, 1998.0, 1999.0, 2000.0, 2001.0, 2002.0, 2003.0, 2004.0, 2005.0, 2006.0, 2007.0, 2008.0, 2009.0, 2010.0, 2011.0, 2012.0, 2013.0, 2016.0, 2017.0, 2018.0]


## Step 6 — Run on all 7 files and merge (fix #3)

Countries only appear in the files where ICO actually tracks them (55 producing countries vs. 35 importing countries). Concatenating with an outer union keeps that distinction intact instead of silently dropping rows via an inner join.

In [8]:
all_long = []
for filename, (metric_name, has_coffee_type) in FILES.items():
    path = INPUT_DIR / filename
    if not path.exists():
        print(f"  [skip] {filename} not found in {INPUT_DIR}")
        continue
    tidy = load_and_tidy(path, metric_name, has_coffee_type)
    print(f"  [ok]   {filename:38s} -> {len(tidy):6d} rows, "
          f"{tidy['Country'].nunique():3d} countries")
    all_long.append(tidy)

combined = pd.concat(all_long, ignore_index=True)
combined = combined.sort_values(["Metric", "Country", "Year"]).reset_index(drop=True)
combined["is_imputed"] = False

print(f"\nCombined: {len(combined):,} total rows across "
      f"{combined['Metric'].nunique()} metrics and "
      f"{combined['Country'].nunique()} countries.")
combined.head()

  [ok]   Coffee_production.csv                  ->   1650 rows,  55 countries
  [ok]   Coffee_domestic_consumption.csv        ->   1650 rows,  55 countries
  [ok]   Coffee_export.csv                      ->   1647 rows,  55 countries
  [ok]   Coffee_import.csv                      ->   1050 rows,  35 countries
  [ok]   Coffee_re_export.csv                   ->   1050 rows,  35 countries
  [ok]   Coffee_importers_consumption.csv       ->   1050 rows,  35 countries
  [ok]   Coffee_green_coffee_inventorie.csv     ->    540 rows,  18 countries

Combined: 8,637 total rows across 7 metrics and 91 countries.


,Country,Coffee type,Metric,Year,value,is_imputed
0,Angola,Robusta/Arabica,domestic_consumption_kg,1990.0,1200000.0,False
1,Angola,Robusta/Arabica,domestic_consumption_kg,1991.0,1800000.0,False
2,Angola,Robusta/Arabica,domestic_consumption_kg,1992.0,2100000.0,False
3,Angola,Robusta/Arabica,domestic_consumption_kg,1993.0,1200000.0,False
4,Angola,Robusta/Arabica,domestic_consumption_kg,1994.0,1500000.0,False


## Step 7 (optional) — Estimate the missing Brazil export values

`NaN` is the honest state, but it's often more useful downstream to have a clearly-flagged estimate. Since Brazil's production is fully intact for 2014/2015/2019, and export closely tracks production in nearby years, a production × ratio estimate is reasonable — as long as it's clearly tagged as an estimate, never confused with real ICO-reported data.

In [9]:
prod_by_year = (
    combined[(combined.Country == "Brazil") & (combined.Metric == "production_kg")]
    .set_index("Year")["value"]
)
exp_by_year = (
    combined[(combined.Country == "Brazil") & (combined.Metric == "export_kg")]
    .set_index("Year")["value"]
)

ratio_years = [y for y in range(2008, 2019) if y in exp_by_year.index and y in prod_by_year.index]
avg_ratio = (exp_by_year.loc[ratio_years] / prod_by_year.loc[ratio_years]).mean()
print(f"Average export/production ratio ({ratio_years[0]}-{ratio_years[-1]}): {avg_ratio:.3f}")

missing_years = [y for y in prod_by_year.index if y not in exp_by_year.index]
estimate_rows = []
for y in missing_years:
    est_value = prod_by_year.loc[y] * avg_ratio
    estimate_rows.append({
        "Country": "Brazil", "Coffee type": np.nan, "Metric": "export_kg_imputed",
        "Year": y, "value": est_value, "is_imputed": True,
    })
    print(f"  Estimated Brazil export {int(y)}: {est_value:,.0f} kg")

combined = pd.concat([combined, pd.DataFrame(estimate_rows)], ignore_index=True)
combined = combined.sort_values(["Metric", "Country", "Year"]).reset_index(drop=True)
print(f"\nTotal rows now: {len(combined):,} (estimates stored as their own "
      f"'export_kg_imputed' metric, flagged is_imputed=True)")

Average export/production ratio (2008-2018): 0.599
  Estimated Brazil export 2014: 1,915,019,363 kg
  Estimated Brazil export 2015: 1,899,427,610 kg
  Estimated Brazil export 2019: 2,091,270,841 kg

Total rows now: 8,640 (estimates stored as their own 'export_kg_imputed' metric, flagged is_imputed=True)


## Step 8 — Save outputs

In [10]:
long_path = OUTPUT_DIR / "ico_coffee_long.csv"
combined.to_csv(long_path, index=False)
print(f"Tidy long-format file: {long_path}  ({len(combined):,} rows)")

for metric_name, _ in FILES.values():
    subset = combined[combined["Metric"] == metric_name]
    if subset.empty:
        continue
    wide = subset.pivot_table(index="Country", columns="Year", values="value", aggfunc="first")
    wide_path = OUTPUT_DIR / f"ico_coffee_wide_{metric_name}.csv"
    wide.to_csv(wide_path)
    print(f"  wide file: {wide_path}  ({wide.shape[0]} countries x {wide.shape[1]} years)")

Tidy long-format file: output\ico_coffee_long.csv  (8,640 rows)
  wide file: output\ico_coffee_wide_production_kg.csv  (55 countries x 30 years)
  wide file: output\ico_coffee_wide_domestic_consumption_kg.csv  (55 countries x 30 years)
  wide file: output\ico_coffee_wide_export_kg.csv  (55 countries x 30 years)
  wide file: output\ico_coffee_wide_import_kg.csv  (35 countries x 30 years)
  wide file: output\ico_coffee_wide_re_export_kg.csv  (35 countries x 30 years)
  wide file: output\ico_coffee_wide_importer_consumption_kg.csv  (35 countries x 30 years)
  wide file: output\ico_coffee_wide_green_coffee_inventory_kg.csv  (18 countries x 30 years)


## Step 9 — Download the results (Colab only)

In [11]:
try:
    from google.colab import files as _colab_files
    _colab_files.download(str(long_path))
except ImportError:
    print(f"Not in Colab — find your files locally in: {OUTPUT_DIR.resolve()}")

Not in Colab — find your files locally in: C:\Users\KrissW Laptop\Downloads\initial data cleaning\output


## Sanity checks

In [12]:
real_rows = combined[~combined["is_imputed"]]

assert not (real_rows["value"] == SENTINEL).any(), "Sentinel value still present in real data!"
print("✅ No sentinel values remain in real (non-imputed) data")

assert (combined["Country"] == combined["Country"].str.strip()).all()
print("✅ No whitespace issues in Country names")

assert combined["Year"].apply(lambda y: float(y).is_integer()).all()
print("✅ Year column is fully numeric")

n_imputed = combined["is_imputed"].sum()
print(f"✅ {n_imputed} imputed row(s), all clearly flagged (Brazil export 2014/2015/2019)")

print(f"\nMetrics found: {sorted(combined['Metric'].unique())}")

✅ No sentinel values remain in real (non-imputed) data
✅ No whitespace issues in Country names
✅ Year column is fully numeric
✅ 3 imputed row(s), all clearly flagged (Brazil export 2014/2015/2019)

Metrics found: ['domestic_consumption_kg', 'export_kg', 'export_kg_imputed', 'green_coffee_inventory_kg', 'import_kg', 'importer_consumption_kg', 'production_kg', 're_export_kg']


## Summary

| Step | What happened |
|---|---|
| Detection | Scanned every year column across all 7 files for `-2147483648`; found 3 cells, all Brazil exports (2014, 2015, 2019) |
| Root cause | `-2147483648` = −2³¹, a classic 32-bit integer overflow / NULL sentinel |
| Recoverable? | No — proved `Total_export` in the raw file was computed *after* the corruption (`reported_total − sum(good years) = 3 × SENTINEL` exactly) |
| Cleaning | Converted to `NaN` inside `load_and_tidy`, applied uniformly to all 7 files |
| Optional estimate | Production × average export/production ratio (2008-2018) — stored as a separate `export_kg_imputed` metric with `is_imputed=True`, never mixed into the real data |
| Other fixes applied | Whitespace-stripped country names (#2), outer-union merge across producer/importer country sets (#3), crop-year → calendar-year normalization (#4) |

Final output: `ico_coffee_long.csv` — one tidy row per `Country / Metric / Year`, with an `is_imputed` flag so downstream analysis never accidentally treats an estimate as ICO-reported fact.